In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import find_peaks

# ============================================================
# Segment-wise Spillover Spike Detection from GFEVD + GGS
# ============================================================

BASE_DIR = Path("./")
RESULT_DIR = Path("./results_segment")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = BASE_DIR / "tvpvar_input_scaled.csv"
INPUT_GFEVD = BASE_DIR / "gfevd_all.npy"
INPUT_SEGMENTS = BASE_DIR / "full_ggs_segments_by_minsize.csv"

DATE_COL = "Date"
TARGET_VAR = "dlog_SOLVPN"

# GGS segment 기준
USE_MIN_SIZE = 120

# Spike detection 설정
Z_K = 2.326
MIN_DISTANCE = 5
LOCAL_PEAK = True
MIN_SEG_LEN = 20

# ============================================================
# Load
# ============================================================

df = pd.read_csv(INPUT_CSV)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

features = [c for c in df.columns if c != DATE_COL]

gfevd = np.load(INPUT_GFEVD)

segments = pd.read_csv(INPUT_SEGMENTS)
segments["start_date"] = pd.to_datetime(segments["start_date"])
segments["end_date"] = pd.to_datetime(segments["end_date"])

T, N, _ = gfevd.shape

if N != len(features):
    raise ValueError("GFEVD dimension and feature count do not match.")

# GFEVD 길이에 맞춰 날짜 보정
dates = df[DATE_COL].iloc[-T:].reset_index(drop=True)

# ============================================================
# Build SOLVPN pairwise spillover series
# convention:
# gfevd[t, i, j] = shock j contribution to variable i
# j -> i
# ============================================================

target_idx = features.index(TARGET_VAR)
records = []

for other in features:
    if other == TARGET_VAR:
        continue

    other_idx = features.index(other)

    solvpn_to_x = gfevd[:, other_idx, target_idx]
    x_to_solvpn = gfevd[:, target_idx, other_idx]
    net_solvpn_to_x = solvpn_to_x - x_to_solvpn

    temp = pd.DataFrame({
        "Date": dates,
        "Pair": f"{TARGET_VAR}__{other}",
        "Source": TARGET_VAR,
        "Target": other,
        "SOLVPN_to_X": solvpn_to_x,
        "X_to_SOLVPN": x_to_solvpn,
        "PairwiseNet_SOLVPN_to_X": net_solvpn_to_x
    })

    records.append(temp)

pairwise_df = pd.concat(records, ignore_index=True)

# ============================================================
# Attach GGS segment ID
# ============================================================

selected_segments = segments[segments["min_size"] == USE_MIN_SIZE].copy()

if selected_segments.empty:
    raise ValueError(f"No segment found for min_size={USE_MIN_SIZE}")

pairwise_df["SegmentID"] = np.nan
pairwise_df["SegmentStart"] = pd.NaT
pairwise_df["SegmentEnd"] = pd.NaT
pairwise_df["SegmentLength"] = np.nan

for _, seg in selected_segments.iterrows():
    mask = (
        (pairwise_df["Date"] >= seg["start_date"]) &
        (pairwise_df["Date"] <= seg["end_date"])
    )

    pairwise_df.loc[mask, "SegmentID"] = int(seg["segment_id"])
    pairwise_df.loc[mask, "SegmentStart"] = seg["start_date"]
    pairwise_df.loc[mask, "SegmentEnd"] = seg["end_date"]
    pairwise_df.loc[mask, "SegmentLength"] = int(seg["length"])

pairwise_df = pairwise_df.dropna(subset=["SegmentID"]).copy()
pairwise_df["SegmentID"] = pairwise_df["SegmentID"].astype(int)

# ============================================================
# Segment-wise spike detection
# ============================================================

spike_records = []

metrics = [
    "SOLVPN_to_X",
    "X_to_SOLVPN",
    "PairwiseNet_SOLVPN_to_X"
]

for (pair, seg_id), sub in pairwise_df.groupby(["Pair", "SegmentID"]):
    sub = sub.sort_values("Date").reset_index(drop=True)

    if len(sub) < MIN_SEG_LEN:
        continue

    for metric in metrics:
        s = sub[metric].astype(float)

        seg_mean = s.mean()
        seg_std = s.std()

        if seg_std == 0 or np.isnan(seg_std):
            continue

        threshold = seg_mean + Z_K * seg_std

        if LOCAL_PEAK:
            peak_idx, _ = find_peaks(
                s.values,
                height=threshold,
                distance=MIN_DISTANCE
            )
            spike_idx = peak_idx
        else:
            spike_idx = np.where(s.values > threshold)[0]

        for idx in spike_idx:
            value = s.iloc[idx]
            z_value = (value - seg_mean) / seg_std

            spike_records.append({
                "Date": sub["Date"].iloc[idx],
                "Pair": pair,
                "Source": sub["Source"].iloc[idx],
                "Target": sub["Target"].iloc[idx],
                "DirectionMetric": metric,
                "SegmentID": seg_id,
                "SegmentStart": sub["SegmentStart"].iloc[idx],
                "SegmentEnd": sub["SegmentEnd"].iloc[idx],
                "SegmentLength": sub["SegmentLength"].iloc[idx],
                "Spillover": value,
                "SegmentMean": seg_mean,
                "SegmentStd": seg_std,
                "SegmentThreshold": threshold,
                "SegmentZ": z_value,
                "SpikeType": "segment"
            })

segment_spikes = pd.DataFrame(spike_records)

if len(segment_spikes) > 0:
    segment_spikes = segment_spikes.sort_values(
        ["Date", "Pair", "DirectionMetric", "SegmentID"]
    ).reset_index(drop=True)

# ============================================================
# Save
# ============================================================

pairwise_out = RESULT_DIR / "solvpn_pairwise_spillover_with_segments.csv"
spike_out = RESULT_DIR / "segmentwise_spillover_spikes_SOLVPN.csv"

pairwise_df.to_csv(pairwise_out, index=False, encoding="utf-8-sig")
segment_spikes.to_csv(spike_out, index=False, encoding="utf-8-sig")

print(f"Saved pairwise with segments: {pairwise_out}")
print(f"Saved segment-wise spikes: {spike_out}")
print("Number of segment-wise spikes:", len(segment_spikes))
print(segment_spikes.head(30))

Saved pairwise with segments: results_segment\solvpn_pairwise_spillover_with_segments.csv
Saved segment-wise spikes: results_segment\segmentwise_spillover_spikes_SOLVPN.csv
Number of segment-wise spikes: 178
         Date                      Pair       Source       Target  \
0  2022-12-07     dlog_SOLVPN__dlog_OIL  dlog_SOLVPN     dlog_OIL   
1  2022-12-12  dlog_SOLVPN__dlog_COPPER  dlog_SOLVPN  dlog_COPPER   
2  2022-12-12     dlog_SOLVPN__dlog_OIL  dlog_SOLVPN     dlog_OIL   
3  2022-12-13    dlog_SOLVPN__d_PJM_LMP  dlog_SOLVPN    d_PJM_LMP   
4  2022-12-13     dlog_SOLVPN__dlog_GAS  dlog_SOLVPN     dlog_GAS   
5  2022-12-13     dlog_SOLVPN__dlog_GAS  dlog_SOLVPN     dlog_GAS   
6  2022-12-13     dlog_SOLVPN__dlog_OIL  dlog_SOLVPN     dlog_OIL   
7  2022-12-13  dlog_SOLVPN__dlog_SILVER  dlog_SOLVPN  dlog_SILVER   
8  2022-12-13  dlog_SOLVPN__dlog_SILVER  dlog_SOLVPN  dlog_SILVER   
9  2022-12-14     dlog_SOLVPN__dlog_DXY  dlog_SOLVPN     dlog_DXY   
10 2022-12-15    dlog_SOLVPN__d_P